# 05c_GMM_nivel2


In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering' / 'nivel2'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'features_tier2.parquet'
TIER_LEVEL = '2'

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object','category']).columns.tolist():
        if out[cat].nunique(dropna=True) > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess(df, nan_threshold_zero=0.30):
    df = df.drop(columns=['user_id']).copy()
    numeric = df.select_dtypes(include=['number']).columns.tolist()
    categorical = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    df = reduce_high_card(df, 10)
    pp = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical),
    ])
    X = pp.fit_transform(df)
    return X, pp, pp.get_feature_names_out()

print(f'TIER {TIER_LEVEL}: cargando {MASTER_PATH.name}')
master = pd.read_parquet(MASTER_PATH)
user_ids = master['user_id'].values
print(f'  shape: {master.shape}')
X, preproc, names = preprocess(master)
joblib.dump(preproc, DATA_MODELS / f'preprocessor_tier{TIER_LEVEL}.joblib')
print(f'  X post-preproc: {X.shape}')
N = len(X)

from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
REPORT = INFORMES_BASE / '05c_gmm'
REPORT.mkdir(parents=True, exist_ok=True)
K_RANGE = list(range(2, 11))
SAMPLE_TRAIN = min(60_000, len(X))
SAMPLE_SIL = 20_000
results = []


TIER 2: cargando features_tier2.parquet
  shape: (21599, 17)
  X post-preproc: (21599, 16)


In [2]:
rng = np.random.RandomState(42)
if len(X) > SAMPLE_TRAIN:
    train_idx = rng.choice(len(X), SAMPLE_TRAIN, replace=False)
    X_train = X[train_idx]
else:
    X_train = X
sil_idx = rng.choice(len(X), min(SAMPLE_SIL, len(X)), replace=False)
X_sil = X[sil_idx]
bics=[]; sils=[]
for k in K_RANGE:
    t0 = time.time()
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, max_iter=200, n_init=1)
    gmm.fit(X_train)
    labels_all = gmm.predict(X)
    labels_sub = gmm.predict(X_sil)
    sil = float(silhouette_score(X_sil, labels_sub)) if len(set(labels_sub))>1 else float('nan')
    db = float(davies_bouldin_score(X, labels_all))
    bic = float(gmm.bic(X_train))
    unique, counts = np.unique(labels_all, return_counts=True)
    max_pct = float(counts.max()/len(labels_all)); min_pct = float(counts.min()/len(labels_all))
    bics.append(bic); sils.append(sil)
    results.append({'algorithm':'GMM','tier':TIER_LEVEL,'k':k,'silhouette':sil,'davies_bouldin':db,'bic':bic,'n_clusters_actual':int(len(set(labels_all))),'max_cluster_pct':max_pct,'min_cluster_pct':min_pct,'elapsed_s':time.time()-t0})
    print(f'  K={k}: sil={sil:.4f} db={db:.4f} bic={bic:.0f} max%={max_pct:.1%} min%={min_pct:.1%}')
    joblib.dump({'model':gmm,'labels':labels_all,'user_ids':user_ids}, DATA_MODELS / f'nivel{TIER_LEVEL}_gmm_k{k}.joblib')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_RANGE, bics, marker='o'); axes[0].set_xlabel('K'); axes[0].set_ylabel('BIC'); axes[0].grid(alpha=0.3); axes[0].set_title(f'BIC nivel{TIER_LEVEL}')
axes[1].plot(K_RANGE, sils, marker='o', color='C2'); axes[1].axhline(0.15, color='red', ls='--', alpha=0.4); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette'); axes[1].grid(alpha=0.3); axes[1].set_title(f'Silhouette nivel{TIER_LEVEL}')
plt.tight_layout(); plt.savefig(REPORT / 'bic_silhouette.png', dpi=120, bbox_inches='tight'); plt.close()
pd.DataFrame(results).to_csv(REPORT / 'metrics.csv', index=False)
print('OK metrics.csv')


  K=2: sil=0.3042 db=1.6964 bic=420555 max%=64.4% min%=35.6%


  K=3: sil=0.3040 db=1.1283 bic=414350 max%=64.4% min%=0.0%


  K=4: sil=0.1032 db=1.3413 bic=180195 max%=60.6% min%=0.0%


  K=5: sil=0.1143 db=1.1186 bic=180994 max%=60.3% min%=0.0%


  K=6: sil=0.1282 db=1.0775 bic=177117 max%=61.0% min%=0.0%


  K=7: sil=0.0960 db=1.0195 bic=171274 max%=60.1% min%=0.0%


  K=8: sil=0.1129 db=1.0852 bic=132355 max%=59.3% min%=0.0%


  K=9: sil=0.1281 db=1.1734 bic=126937 max%=58.6% min%=0.0%


  K=10: sil=0.1134 db=1.1192 bic=99941 max%=57.8% min%=0.0%
OK metrics.csv
